In [63]:
import faostat as fao

In [2]:
all_datasets=fao.list_datasets_df() # 查看所有的数据集

In [2]:
# 这里需要的数据集的code为TCL，因此查看此数据集下的pars
Pars=fao.list_pars("FBSH")
Pars

['area', 'element', 'item', 'year']

In [5]:
# 可以查看一个par下面有哪些项 首先element
element=fao.get_par("FBSH","element")
element_code=["2520","2141","2120","2151","2130","2525"]
element_code

['2520', '2141', '2120', '2151', '2130', '2525']

In [6]:
# year下面的项
years=fao.get_par("FBSH","years");years

{'2013': '2013',
 '2012': '2012',
 '2011': '2011',
 '2010': '2010',
 '2009': '2009',
 '2008': '2008',
 '2007': '2007',
 '2006': '2006',
 '2005': '2005',
 '2004': '2004',
 '2003': '2003',
 '2002': '2002',
 '2001': '2001',
 '2000': '2000',
 '1999': '1999',
 '1998': '1998',
 '1997': '1997',
 '1996': '1996',
 '1995': '1995',
 '1994': '1994',
 '1993': '1993',
 '1992': '1992',
 '1991': '1991',
 '1990': '1990',
 '1989': '1989',
 '1988': '1988',
 '1987': '1987',
 '1986': '1986',
 '1985': '1985',
 '1984': '1984',
 '1983': '1983',
 '1982': '1982',
 '1981': '1981',
 '1980': '1980',
 '1979': '1979',
 '1978': '1978',
 '1977': '1977',
 '1976': '1976',
 '1975': '1975',
 '1974': '1974',
 '1973': '1973',
 '1972': '1972',
 '1971': '1971',
 '1970': '1970',
 '1969': '1969',
 '1968': '1968',
 '1967': '1967',
 '1966': '1966',
 '1965': '1965',
 '1964': '1964',
 '1963': '1963',
 '1962': '1962',
 '1961': '1961'}

In [6]:
# area下面的项，这里不看整个area了，直接看countries
countries=fao.get_par("FBSH","countries")
countries_code=list(countries.values()) # 把国家代码取出

In [7]:
items=fao.get_par("FBSH","items");items

{'Population': '2501',
 'Wheat and products': '2511',
 'Rice (Milled Equivalent)': '2805',
 'Barley and products': '2513',
 'Maize and products': '2514',
 'Rye and products': '2515',
 'Oats': '2516',
 'Millet and products': '2517',
 'Sorghum and products': '2518',
 'Cereals, Other': '2520',
 'Cassava and products': '2532',
 'Potatoes and products': '2531',
 'Sweet potatoes': '2533',
 'Yams': '2535',
 'Roots, Other': '2534',
 'Sugar cane': '2536',
 'Sugar beet': '2537',
 'Sugar non-centrifugal': '2541',
 'Sugar (Raw Equivalent)': '2542',
 'Sweeteners, Other': '2543',
 'Honey': '2745',
 'Beans': '2546',
 'Peas': '2547',
 'Pulses, Other and products': '2549',
 'Nuts and products': '2551',
 'Soyabeans': '2555',
 'Groundnuts (Shelled Eq)': '2556',
 'Sunflower seed': '2557',
 'Rape and Mustardseed': '2558',
 'Cottonseed': '2559',
 'Coconuts - Incl Copra': '2560',
 'Sesame seed': '2561',
 'Palm kernels': '2562',
 'Olives (including preserved)': '2563',
 'Oilcrops, Other': '2570',
 'Soyabean O

In [8]:
# item下面的项,注意这里要加s，items是item的子集,也就是不要itemsagg
items=fao.get_par("FBSH","items")
# 取出population,将这个下面所有的项用population列表表示
population = ["Population"]
population_code=[items[key] for key in population] #把项的代码取出

In [9]:
# 取出除人口的其它代码，
others_code =list(set(list(items.values()))-set(population_code))

In [10]:
# 模仿网页筛选取数据

data=fao.get_data_df("TCL",pars={"area":countries_code,"element":2610,"item":Crops_and_livestock_code,"year":1961},show_flags=True, null_values=True)

In [11]:
# 函数定义，后面可用来循环提取不同年份
def FBSH_sort(area_code,element_code,items_code,year):
    # year输入0代表取所有年份的
    if year==0:
        data=fao.get_data_df("FBSH",pars={"area":area_code,"element":element_code,"item":items_code},show_flags=True, null_values=True)
    else:
        data=fao.get_data_df("FBSH",pars={"area":area_code,"element":element_code,"item":items_code,"year":year},show_flags=True, null_values=True)
    return data

In [12]:
# 提取数据的方案一：一年年的循环访问url进行筛选提取
# 提取1961年到2013的，import,crop
for year in range(1989,2014):
    data_year=FBSH_sort(countries_code,element_code,others_code,year)
    data_year.to_csv(str(year)+".csv")

In [ ]:
# 提取数据的方案二：先一次性把所有年份的数据获取下来，通过pandas库操作再按年份进行分类
data_all_year=FBSH_sort(countries_code,element_code,others_code,0)
# 按年份分割并保存数据
for year, group in data_all_year.groupby('Year'):
    group.to_csv(f"{year}.csv")

In [73]:
# 方案二好处是减少url访问次数，防止频繁，坏处是要读取所有年份的，所有数据量可能会过大,方案一优缺点与方案二相反
# 如果数据量太大，无法一次性取出就用方案一，否则方案二
# 能一次性取出来就用方案二，推荐！，时间最短

In [1]:
import numpy as np


In [24]:
a=np.array([[1,2,21,1,1],[1,1,1]])

C:\Users\typing\AppData\Local\Temp\ipykernel_8612\3085770710.py:1: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  a=np.array([[1,2,21,1,1],[1,1,1]])


In [11]:
np.nan

nan

In [12]:
np.arange(10)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [13]:
range(10)

range(0, 10)

In [14]:
print(list(range(10)))


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [15]:
np.arange(1,10,0.5)

array([1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5, 5. , 5.5, 6. , 6.5, 7. ,
       7.5, 8. , 8.5, 9. , 9.5])

In [25]:
a.shape

(2,)

In [26]:
import pandas as pd

In [32]:
S1=pd.Series([1,1])
S2=pd.Series([3,3])
pd.concat([S1,S2],axis=0)

0    1
1    1
0    3
1    3
dtype: int64

In [34]:
pd.Series([[1,1],[5,5]])

0    [1, 1]
1    [5, 5]
dtype: object

In [35]:
a=[1,2,3,4,5,6,7,8,9]

In [52]:
a[1:9:-1]

[]

In [59]:
df=pd.DataFrame({"测试":[1,2,3,4,5,3,2]},index=[2,6,1,9,0,3,2])

In [62]:
df["测试"]

2    1
6    2
1    3
9    4
0    5
3    3
2    2
Name: 测试, dtype: int64

In [67]:
# 模仿网页筛选取数据

data=fao.get_data_df("TCL",pars={"element":2610,"year":1961},show_flags=True, null_values=True)

In [68]:
data.isnull().sum()

Domain Code              0
Domain                   0
Area Code (FAO)          0
Area                     0
Element Code             0
Element                  0
Item Code                0
Item                     0
Year Code                0
Year                     0
Unit                126616
Value               126616
Flag                126616
Flag Description    126616
Note                     0
dtype: int64

In [81]:
data.sort_values(by="Item Code",ascending=False).head(5)

,Domain Code,Domain,Area Code (FAO),Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
24073,TCL,Crops and livestock products,128,"China, Macao SAR",5610,Import Quantity,999,"Raw hides and skins of sheep or lambs, with wool",1961,1961,None,None,None,None,
143379,TCL,Crops and livestock products,52030,Northern America (excluding intra-trade),5610,Import Quantity,999,"Raw hides and skins of sheep or lambs, with wool",1961,1961,None,None,None,None,
99357,TCL,Crops and livestock products,188,Saint Kitts and Nevis,5610,Import Quantity,999,"Raw hides and skins of sheep or lambs, with wool",1961,1961,None,None,None,None,
76389,TCL,Crops and livestock products,137,Mauritius,5610,Import Quantity,999,"Raw hides and skins of sheep or lambs, with wool",1961,1961,None,None,None,None,
64905,TCL,Crops and livestock products,118,Kuwait,5610,Import Quantity,999,"Raw hides and skins of sheep or lambs, with wool",1961,1961,None,None,None,None,


In [79]:
data["Item Code"].dtype

dtype('O')

In [82]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178640 entries, 0 to 178639
Data columns (total 15 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Domain Code       178640 non-null  object
 1   Domain            178640 non-null  object
 2   Area Code (FAO)   178640 non-null  object
 3   Area              178640 non-null  object
 4   Element Code      178640 non-null  object
 5   Element           178640 non-null  object
 6   Item Code         178640 non-null  object
 7   Item              178640 non-null  object
 8   Year Code         178640 non-null  object
 9   Year              178640 non-null  object
 10  Unit              52024 non-null   object
 11  Value             52024 non-null   object
 12  Flag              52024 non-null   object
 13  Flag Description  52024 non-null   object
 14  Note              178640 non-null  object
dtypes: object(15)
memory usage: 20.4+ MB


In [91]:
data[["Note","Item Code"]].tail(1)

,Note,Item Code
178639,,1881


In [92]:
data.index

RangeIndex(start=0, stop=178640, step=1)

In [93]:
data.shape

(178640, 15)

In [107]:
data.loc[0:5,"Area":"Domain"]

""
0
1
2
3
4
5


In [112]:
data.set_index("Area",inplace=True)

In [113]:
data.index

Index(['Afghanistan', 'Afghanistan', 'Afghanistan', 'Afghanistan',
       'Afghanistan', 'Afghanistan', 'Afghanistan', 'Afghanistan',
       'Afghanistan', 'Afghanistan',
       ...
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)',
       'Net Food Importing Developing Countries (excluding intra-trade)'],
      dtype='object', name='Area', length=178640)

In [117]:
import numpy as np
a=np.array([0,1,2,3,4])
c=np.array([0,1,2,3,4])
import pandas as pd
b=pd.Series([0,1,2,3,4])

In [116]:
a+b

0    0
1    2
2    4
3    6
4    8
dtype: int64

In [118]:
a+c

array([0, 2, 4, 6, 8])

In [120]:
k=np.array([[1,2],[2,3]])

In [127]:
np.diag(k).sum()

4

In [134]:
n1=np.arange(9).reshape((3,3))

In [135]:
n2=np.linspace(10,20,9).reshape((3,3))

In [133]:
pd.DataFrame(np.arange(9).reshape((3,3)))

,0,1,2
0,0,1,2
1,3,4,5
2,6,7,8


In [140]:
nv=np.vstack((n1,n2))

In [139]:
nh=np.hstack((n1,n2))

In [149]:
np.vsplit(nv,3)

[array([[0., 1., 2.],
        [3., 4., 5.]]),
 array([[ 6.  ,  7.  ,  8.  ],
        [10.  , 11.25, 12.5 ]]),
 array([[13.75, 15.  , 16.25],
        [17.5 , 18.75, 20.  ]])]

In [151]:
data.reset_index(drop=True, inplace=True)

In [154]:
data[data["Item Code"]=="1168"]

,Domain Code,Domain,Area Code (FAO),Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
4,TCL,Crops and livestock products,2,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
642,TCL,Crops and livestock products,3,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
1280,TCL,Crops and livestock products,4,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,t,0,A,Official figure,
1918,TCL,Crops and livestock products,7,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
2556,TCL,Crops and livestock products,8,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175454,TCL,Crops and livestock products,58030,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
176092,TCL,Crops and livestock products,5815,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,t,785,A,Official figure,
176730,TCL,Crops and livestock products,58150,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,None,None,None,None,
177368,TCL,Crops and livestock products,5817,5610,Import Quantity,1168,Animal oils and fats n.e.c.,1961,1961,t,1966,A,Official figure,
